In [ ]:
import pandas as pd
from datetime import datetime, timedelta
from openlocationcode import openlocationcode as olc

dataset = "NYC"

def do_filter(df, poi_min_freq=10, user_min_freq=10):
    df = df.copy()
    df['PoiFreq'] = df.groupby('Pid')['Uid'].transform('count')
    df = df[df['PoiFreq'] >= poi_min_freq]
    df['UserFreq'] = df.groupby('Uid')['Pid'].transform('count')
    df = df[df['UserFreq'] >= user_min_freq]
    df = df.drop(columns=['PoiFreq', 'UserFreq'])
    return df

def get_pluscode(latitude, longitude):
    plus_code = olc.encode(latitude, longitude)
    return plus_code[:6]

file_name = f"datasets/{dataset}.txt"
df = pd.read_csv(file_name, sep="\t", encoding='latin-1', header=None, names=[
    "User ID", "Venue ID", "Venue Category ID", "Venue Category Name", "Latitude", "Longitude", "Timezone Offset", "UTC Time"
])

df["Region"] = df.apply(lambda row: get_pluscode(row['Latitude'], row['Longitude']), axis=1)
df["UTC Time"] = pd.to_datetime(df["UTC Time"], format="%a %b %d %H:%M:%S %z %Y")
df["Local Time"] = (df["UTC Time"] + df["Timezone Offset"].apply(lambda x: timedelta(minutes=x))).dt.strftime("%Y-%m-%d %H:%M")
df.columns = ["Uid", "Pid", "Venue Category ID", "Catname", "Lat", "Lon", "Timezone Offset", "UTC Time", "Region", "Time"]
df = df[["Uid", "Pid", "Catname", "Region", "Time"]]
filtered_df = do_filter(df, poi_min_freq=10, user_min_freq=10)

datapath = file_name.split(".")[0] + "/"
outname = file_name.split("/")[-1].split(".")[0]
filtered_df.to_csv(f"{datapath}{outname}.csv", index=False)

In [ ]:
import pandas as pd
import random
import os


dataset = "NYC"
df = pd.read_csv(f"datasets/{dataset}/{dataset}.csv")
uids = list(df["Uid"].unique())
pids = list(df["Pid"].unique())
cats = list(df["Catname"].unique())
regs = list(df["Region"].unique())

random.shuffle(uids)
random.shuffle(pids)
random.shuffle(cats)
random.shuffle(regs)

uid_map = {uid: i for i, uid in enumerate(uids, start=1)}
pid_map = {pid: i for i, pid in enumerate(pids, start=1)}
cat_map = {cat: i for i, cat in enumerate(cats, start=1)}
reg_map = {reg: i for i, reg in enumerate(regs, start=1)}

df["Uid"] = df["Uid"].map(uid_map)
df["Pid"] = df["Pid"].map(pid_map)
df["Catname"] = df["Catname"].map(cat_map)
df["Region"] = df["Region"].map(reg_map)

if not os.path.exists(f"datasets/{dataset}"):
    os.makedirs(f"datasets/{dataset}")

pd.DataFrame(list(uid_map.items()), columns=["Original_Uid", "Mapped_Uid"]).to_csv(f"datasets/{dataset}/uid_mapping.csv", index=False)
pd.DataFrame(list(pid_map.items()), columns=["Original_Pid", "Mapped_Pid"]).to_csv(f"datasets/{dataset}/pid_mapping.csv", index=False)
pd.DataFrame(list(cat_map.items()), columns=["Original_Catname", "Mapped_Catname"]).to_csv(f"datasets/{dataset}/catname_mapping.csv", index=False)
pd.DataFrame(list(reg_map.items()), columns=["Original_Region", "Mapped_Region"]).to_csv(f"datasets/{dataset}/region_mapping.csv", index=False)

df.to_csv(f"datasets/{dataset}/data.csv", index=False)

映射完成，已保存文件。


In [ ]:
def split_data(datafold, train_ratio=0.8, valid_ratio=0.1, test_ratio=0.1):
    file_name = f"{datafold}/{datafold}.csv"
    df = pd.read_csv(file_name)
    df = df[['Uid', 'Pid', 'Time']]
    df = df.sort_values(by='Time')
    train_size = int(train_ratio * len(df))
    valid_size = int(valid_ratio * len(df))
    
    train_df = df[:train_size]
    valid_df = df[train_size:train_size + valid_size]
    test_df = df[train_size + valid_size:]

    def remove_users_pois_test(df_train, df_test):
        users_train = df_train['Uid'].unique()
        pois_train = df_train['Pid'].unique()
        df_test = df_test[df_test['Uid'].isin(users_train)]
        df_test = df_test[df_test['Pid'].isin(pois_train)]
        return df_test

    train_df.to_csv(f'datasets/{dataset}/train_data.csv', index=False)

    valid_df = remove_users_pois_test(train_df, valid_df)
    valid_uids = valid_df['Uid'].unique()
    expanded_valid_df = df[df['Uid'].isin(valid_uids)]
    expanded_valid_df.to_csv(f'datasets/{dataset}/valid_data.csv', index=False)

    test_df = remove_users_pois_test(train_df, test_df)
    test_uids = test_df['Uid'].unique()
    expanded_test_df = df[df['Uid'].isin(test_uids)]
    expanded_test_df.to_csv(f'datasets/{dataset}/test_data.csv', index=False)

dataset = "NYC"
split_data(dataset, train_ratio=0.8, valid_ratio=0.1)


In [ ]:
import pandas as pd
from collections import Counter, defaultdict

dataset = "NYC"
file_name = f"datasets/{dataset}/train_data.csv"
df = pd.read_csv(file_name)
df["Time"] = pd.to_datetime(df["Time"]).dt.hour
poi_sequence = df.groupby("Uid").agg({
    "Pid": list,
    "Catname": list
}).reset_index()

def get_forward_neighbors(df, column, min_freq=1):
    neighbor_counts = defaultdict(Counter)
    all_pois = set()

    for sequence in df[column]:
        all_pois.update(sequence)
        for i in range(len(sequence) - 1):
            current_poi = sequence[i]
            next_poi = sequence[i + 1]
            neighbor_counts[current_poi][next_poi] += 1

    df_data = []
    for poi in all_pois:
        counter = neighbor_counts.get(poi, {})
        filtered_neighbors = {
            neighbor: freq for neighbor, freq in counter.items() if freq >= min_freq
        }
        if filtered_neighbors:
            sorted_neighbors = [
                neighbor for neighbor, _ in sorted(filtered_neighbors.items(), key=lambda x: x[1], reverse=True)
            ]
        else:
            sorted_neighbors = []
        df_data.append((poi, sorted_neighbors))

    neighbors_df = pd.DataFrame(df_data, columns=[column, "neighbors"])
    return neighbors_df

def get_neighbors(df, column, min_freq=1):
    neighbor_counts = defaultdict(Counter)
    all_pois = set()

    for sequence in df[column]:
        all_pois.update(sequence)
        for i, poi in enumerate(sequence):
            if i > 0:  
                neighbor_counts[poi][sequence[i - 1]] += 1
            if i < len(sequence) - 1:  
                neighbor_counts[poi][sequence[i + 1]] += 1

    df_data = []
    for poi in all_pois:
        counter = neighbor_counts.get(poi, {})
        filtered_neighbors = {
            neighbor: freq for neighbor, freq in counter.items() if freq >= min_freq
        }
        sorted_neighbors = [
            neighbor for neighbor, _ in sorted(filtered_neighbors.items(), key=lambda x: x[1], reverse=True)
        ]
        df_data.append((poi, sorted_neighbors))

    neighbors_df = pd.DataFrame(df_data, columns=[column, "neighbors"])
    return neighbors_df

poi_info = df.groupby("Pid").agg({
    "Uid": list,
    "Catname": lambda x: x.iloc[0],
    "Region": lambda x: x.iloc[0],
    "Time": list
}).reset_index()

poi_info["Uid"] = poi_info["Uid"].apply(lambda uids: [uid for uid, count in Counter(uids).items() if count >= 1])

poi_info["Time"] = poi_info["Time"].apply(lambda times: [time for time, count in Counter(times).items() if count >= 1])


poi_neighbors = get_neighbors(poi_sequence,"Pid", 1)
poi_info["neighbors"] = poi_info["Pid"].map(poi_neighbors.set_index("Pid")["neighbors"])
forward_neighbors = get_forward_neighbors(poi_sequence,"Pid", 1)
poi_info["forward_neighbors"] = poi_info["Pid"].map(forward_neighbors.set_index("Pid")["neighbors"])
poi_info.to_csv(f"datasets/{dataset}/poi_info.csv", index=False)
